# Práctica 4.1 — Contrato del artefacto y manifiesto

Daniel Alonso · Operación de Modelos · 22/10/2026

`->` Quiero poder inspeccionar la compatibilidad antes de deserializar el modelo de S3. En este lienzo diseño el bundle; las funciones de guardado, carga e inferencia se implementan en el starter de la clase 2.

`~>` Conservo las once features, las categorías y el preprocesado de S3. No entreno un modelo nuevo ni añado una API.

[Informe de la semana](../../../../../../REPORT.md) · [Lienzo y matriz](LIENZO.md)


In [1]:
from pathlib import Path
import csv
import json

week_root = next(
    parent for parent in (Path.cwd(), *Path.cwd().parents)
    if (parent / "modules/04-model-packaging").is_dir()
)
input_path = week_root.parent / "semana3/assets/03-wine-quality/inference_samples.csv"
with input_path.open(newline="", encoding="utf-8") as source:
    reader = csv.DictReader(source)
    rows = list(reader)
    columns = reader.fieldnames
feature_names = [name for name in columns if name != "sample_id"]
assert len(rows) == 5 and len(feature_names) == 11
print("Fuente única: semana3/assets/03-wine-quality/inference_samples.csv")
print("Muestras:", len(rows))
print("Features en orden:", feature_names)
print("sample_id es un metadato: no entra en el vector del modelo.")


Fuente única: semana3/assets/03-wine-quality/inference_samples.csv
Muestras: 5
Features en orden: ['fixed_acidity', 'volatile_acidity', 'citric_acid', 'residual_sugar', 'chlorides', 'free_sulfur_dioxide', 'total_sulfur_dioxide', 'density', 'ph', 'sulphates', 'alcohol']
sample_id es un metadato: no entra en el vector del modelo.


## Inspección del payload legado

`>` Inspecciono exclusivamente el artefacto local de confianza que utilicé en S3. Compruebo las claves y sus tipos; todavía no hago predicciones.

`~>` El notebook también puede repetirse sin ese binario: en ese caso utiliza un `DummyClassifier` pequeño y declara una versión de demostración. El resultado guardado en esta entrega procede del Random Forest real. El fallback demuestra el contrato y no pretende reproducir sus predicciones.


In [2]:
import joblib
from sklearn.dummy import DummyClassifier

starter = week_root / (
    "modules/04-model-packaging/exercises/01-model-packaging/"
    "01.02-serializable-inference-module/problem/starter"
)
legacy_path = starter / "models/wine_quality_classifier.joblib"
if legacy_path.is_file():
    # Solo abro el modelo del material de clase, ya utilizado en mi práctica 3.
    legacy_payload = joblib.load(legacy_path)
    artifact_source = "Artefacto real de S3 (local, de confianza)"
else:
    demo = DummyClassifier(strategy="constant", constant="acceptable")
    demo.fit([[0.0] * len(feature_names)], ["acceptable"])
    legacy_payload = {
        "estimator": demo,
        "feature_names": feature_names,
        "model_version": "wine-quality-contract-demo-v1",
    }
    artifact_source = "Demostración Dummy; no es el Random Forest de S3"

assert set(legacy_payload) == {"estimator", "feature_names", "model_version"}
assert legacy_payload["feature_names"] == feature_names
print(artifact_source)
print("Claves y tipos:", {key: type(value).__name__ for key, value in legacy_payload.items()})
print("Tipo real:", type(legacy_payload["estimator"]).__name__)
print("Versión:", legacy_payload["model_version"])
print("Metadatos ausentes en el payload legado:",
      ["schema_version", "preprocessing_version", "output_labels", "estimator_type"])


Artefacto real de S3 (local, de confianza)
Claves y tipos: {'estimator': 'RandomForestClassifier', 'feature_names': 'list', 'model_version': 'str'}
Tipo real: RandomForestClassifier
Versión: wine-quality-rf-v1
Metadatos ausentes en el payload legado: ['schema_version', 'preprocessing_version', 'output_labels', 'estimator_type']


## Qué guardo y por qué

| Dato | Destino | Justificación y riesgo si falta |
| --- | --- | --- |
| Estimador ajustado | `model.joblib`, clave `estimator` | Conservo el objeto y sus parámetros aprendidos; no vuelvo a entrenarlo al predecir. |
| `schema_version` | `manifest.json` | Puedo rechazar un formato que mi cargador desconoce antes de abrir el binario. |
| `model_version` | `manifest.json` | Identifico qué estimador produjo la respuesta. No deduzco la versión del nombre del archivo. |
| `preprocessing_version` | `manifest.json`; transformación en código | Bloqueo una representación diferente aunque tenga once números. |
| `feature_names` | `manifest.json`; orden canónico en código | Comparo nombres y posiciones con S3; mezclar densidad y alcohol altera el significado del vector. |
| `output_labels` | `manifest.json`; contrato de salida en código | Limito las categorías que el consumidor puede recibir. |
| `estimator_type` | `manifest.json`; clase real en el binario | Declaro el tipo esperado y lo contrasto tras cargarlo. |
| Rangos, ausentes, campos extra e interfaz `predict`/`predict_proba` | Código de contratos y carga | El JSON describe el contrato, pero el código debe hacerlo cumplir. |
| Preprocesado y política de filas | Código | Construyo el mismo vector y publico el resultado únicamente si todo el lote es válido. |

`~>` No mezclo `model_version` (identidad del estimador), `preprocessing_version` (representación) y `schema_version` (forma del bundle).

## Manifiesto propuesto

`>` Declaro seis campos obligatorios, sin valores por defecto al leer un bundle y sin campos extra. Solo el helper de creación compone esos campos desde el contrato actual y una versión de modelo explícita.


In [3]:
manifest_draft = {
    "schema_version": "wine-quality-bundle-v1",
    "model_version": legacy_payload["model_version"],
    "preprocessing_version": "wine-red-features-v1",
    "feature_names": feature_names,
    "output_labels": ["needs_review", "acceptable", "excellent"],
    "estimator_type": type(legacy_payload["estimator"]).__name__,
}
required_keys = {
    "schema_version", "model_version", "preprocessing_version",
    "feature_names", "output_labels", "estimator_type",
}
assert set(manifest_draft) == required_keys
assert manifest_draft["feature_names"] == legacy_payload["feature_names"]
assert "sample_id" not in manifest_draft["feature_names"]
assert callable(getattr(legacy_payload["estimator"], "predict", None))
assert callable(getattr(legacy_payload["estimator"], "predict_proba", None))
print(json.dumps(manifest_draft, indent=2))


{
  "schema_version": "wine-quality-bundle-v1",
  "model_version": "wine-quality-rf-v1",
  "preprocessing_version": "wine-red-features-v1",
  "feature_names": [
    "fixed_acidity",
    "volatile_acidity",
    "citric_acid",
    "residual_sugar",
    "chlorides",
    "free_sulfur_dioxide",
    "total_sulfur_dioxide",
    "density",
    "ph",
    "sulphates",
    "alcohol"
  ],
  "output_labels": [
    "needs_review",
    "acceptable",
    "excellent"
  ],
  "estimator_type": "RandomForestClassifier"
}


## Invariantes que bloquearán la ejecución

1. El JSON tiene las seis claves obligatorias y ninguna desconocida; `schema_version` es `wine-quality-bundle-v1`.
2. `preprocessing_version` coincide con `wine-red-features-v1`; `feature_names` coincide exactamente en nombres, longitud y orden con S3.
3. `output_labels` conserva `needs_review`, `acceptable` y `excellent`; la salida debe pertenecer a ese conjunto y tener confianza finita en [0, 1].
4. `model_version` y `estimator_type` son texto no vacío. Después de deserializar compruebo el tipo real y que `predict` y `predict_proba` son invocables.
5. Si existen `n_features_in_` y `classes_`, son coherentes con once features y las categorías permitidas. No uso el orden de `output_labels` para indexar probabilidades: consulto `classes_`.
6. Todos los inputs y todas las respuestas son válidos antes de publicar el CSV.

`>>` La siguiente matriz describe el fallo esperado. Los tests de la clase 2 comprueban estas fronteras con artefactos temporales; esta celda no implementa el cargador.

| Cambio | Comprobación | Error esperado | Etapa de rechazo | ¿CSV publicado? |
| --- | --- | --- | --- | --- |
| Intercambiar `density` y `alcohol` en `feature_names` del manifiesto | Comparación de la secuencia completa | Orden de features incompatible | Validación del JSON, antes de Joblib | No |
| Declarar `wine-red-features-v2` | Igualdad con el preprocesado soportado | Versión no soportada | Validación del JSON, antes de Joblib | No |
| Declarar `low` en `output_labels` | Categorías del contrato | Etiqueta desconocida | Validación del JSON, antes de Joblib | No |
| Declarar RF y guardar un Dummy | Tipo real frente a `estimator_type` | Tipo de estimador incompatible | Tras deserializar, antes de inferir | No |
| Estimador sin `predict_proba` | Interfaz mínima invocable | Método requerido ausente | Tras deserializar, antes de inferir | No |
| Modelo devuelve `unknown` | Contrato de respuesta | Categoría no permitida | Inferencia, antes de escribir | No |
| Segunda muestra con `ph=99` | Rango [2.5, 4.5] | Campo fuera de rango | Validación de filas, antes de cargar el bundle | No |

`~>` El primer caso modifica el orden semántico del vector declarado en el manifiesto. Reordenar solo las columnas del CSV conservando sus nombres no cambia el vector: el preprocesado accede por nombre y fija las posiciones.


## Dos entradas inválidas comprobadas

`>` Pruebo una medición fuera de rango y un dato obligatorio ausente con el contrato preparado por el profesor. No abro ningún binario en esta comprobación.


In [4]:
from pydantic import ValidationError
from model_packaging.contracts import WineQualityRequest

valid_sample = {key: value for key, value in rows[0].items() if key != "sample_id"}
invalid_range = {**valid_sample, "ph": "99"}
missing_alcohol = {key: value for key, value in valid_sample.items() if key != "alcohol"}
for case_name, sample in [("ph fuera de rango", invalid_range),
                          ("alcohol ausente", missing_alcohol)]:
    try:
        WineQualityRequest.model_validate(sample)
    except ValidationError as error:
        print(case_name, "-> rechazo en validación de entrada:",
              [(item["loc"], item["type"]) for item in error.errors()])
    else:
        raise AssertionError("La entrada inválida no debería aceptarse")


ph fuera de rango -> rechazo en validación de entrada: [(('ph',), 'less_than_equal')]
alcohol ausente -> rechazo en validación de entrada: [(('alcohol',), 'missing')]


## Política de errores y paso a la clase 2

`~>` Si la segunda fila falla, rechazo el lote completo, comunico la fila y el campo y no publico un CSV nuevo. Si había una salida anterior, la conservo. Primero valido todas las filas, luego cargo el bundle, infiero y valido todas las respuestas; por último escribo un temporal junto al destino y hago un reemplazo atómico.

`>` Orden previsto: manifiesto → guardado → carga con comprobaciones → salida validada → CLI → caso de fallo adicional. Reutilizo `contracts.py` y `preprocess.py` sin cambiarlos. Los tests constituyen la especificación ejecutable.

`>>` Conservo el [manifiesto propuesto](../../../../../../entrega/manifest_propuesto.json), la [matriz en CSV](../../../../../../entrega/matriz_compatibilidad.csv) y el [lienzo](LIENZO.md). La evidencia ejecutable de las decisiones está en los tests y en el informe de clase 2.

`=>` El diseño permite comprobar la compatibilidad antes de deserializar, pero no autentica el binario ni garantiza su seguridad. Solo cargo Joblib de procedencia confiable. No presento como realizada una revisión por otra pareja.
